In [7]:
%%writefile konfigurasi.py
import os

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
NAMA_DB = 'pengeluaran_harian.db'
DB_PATH = os.path.join(BASE_DIR, NAMA_DB)
KATEGORI_PENGELUARAN = ["Makanan", "Transportasi", "Hiburan", "Tagihan", "Belanja", "Kesehatan", "Pendidikan", "Lainnya"]
KATEGORI_DEFAULT = "Lainnya"

Overwriting konfigurasi.py


In [8]:
%%writefile setup_db_pengeluaran.py
import sqlite3
import os
from konfigurasi import DB_PATH

def setup_database():
    print(f"Memeriksa/membuat database di: {DB_PATH}")
    conn = None
    try:
        conn = sqlite3.connect(DB_PATH)
        cursor = conn.cursor()
        sql_create_table = """
        CREATE TABLE IF NOT EXISTS transaksi (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            deskripsi TEXT NOT NULL,
            jumlah REAL NOT NULL CHECK (jumlah > 0),
            kategori TEXT,
            tanggal DATE NOT NULL
        );"""
        cursor.execute(sql_create_table)
        conn.commit()
        print(" -> Tabel 'transaksi' siap.")
        return True
    except sqlite3.Error as e:
        print(f" -> Error SQLite saat setup: {e}")
        return False
    finally:
        if conn: conn.close()

if __name__ == "__main__":
    setup_database()

Overwriting setup_db_pengeluaran.py


In [9]:
%%writefile database.py
import sqlite3
import pandas as pd
from konfigurasi import DB_PATH

def get_db_connection() -> sqlite3.Connection | None:
    try:
        conn = sqlite3.connect(DB_PATH, timeout=10, detect_types=sqlite3.PARSE_DECLTYPES)
        conn.row_factory = sqlite3.Row
        return conn
    except sqlite3.Error as e:
        print(f"ERROR: {e}")
        return None

def execute_query(query: str, params: tuple = None):
    conn = get_db_connection()
    if not conn: return None
    last_id = None
    try:
        cursor = conn.cursor()
        if params: cursor.execute(query, params)
        else: cursor.execute(query)
        conn.commit()
        return cursor.lastrowid
    except sqlite3.Error as e:
        conn.rollback()
        return None
    finally:
        if conn: conn.close()

def fetch_query(query: str, params: tuple = None, fetch_all: bool = True):
    conn = get_db_connection()
    if not conn: return None
    try:
        cursor = conn.cursor()
        if params: cursor.execute(query, params)
        else: cursor.execute(query)
        return cursor.fetchall() if fetch_all else cursor.fetchone()
    finally:
        if conn: conn.close()

def get_dataframe(query: str, params: tuple = None) -> pd.DataFrame:
    conn = get_db_connection()
    if not conn: return pd.DataFrame()
    try:
        return pd.read_sql_query(query, conn, params=params)
    finally:
        if conn: conn.close()

def setup_database_initial():
    conn = get_db_connection()
    if not conn: return False
    try:
        cursor = conn.cursor()
        cursor.execute("""
        CREATE TABLE IF NOT EXISTS transaksi (
            id INTEGER PRIMARY KEY AUTOINCREMENT, 
            deskripsi TEXT NOT NULL,
            jumlah REAL NOT NULL CHECK (jumlah > 0), 
            kategori TEXT,
            tanggal DATE NOT NULL
        );""")
        conn.commit()
        return True
    except: return False
    finally:
        if conn: conn.close()

Overwriting database.py


In [10]:
%%writefile model.py
import datetime

class Transaksi:
    def __init__(self, deskripsi: str, jumlah: float, kategori: str, tanggal: datetime.date | str, id_transaksi: int | None = None):
        self.id = id_transaksi
        self.deskripsi = str(deskripsi) if deskripsi else "Tanpa Deskripsi"
        try:
            self.jumlah = float(jumlah) if float(jumlah) > 0 else 0.0
        except: self.jumlah = 0.0
        self.kategori = str(kategori) if kategori else "Lainnya"
        
        if isinstance(tanggal, datetime.date): self.tanggal = tanggal
        elif isinstance(tanggal, str):
            try: self.tanggal = datetime.datetime.strptime(tanggal, "%Y-%m-%d").date()
            except: self.tanggal = datetime.date.today()
        else: self.tanggal = datetime.date.today()

Overwriting model.py


In [11]:
%%writefile manajer_anggaran.py
import datetime
import pandas as pd
from model import Transaksi
import database

class AnggaranHarian:
    _db_setup_done = False

    def __init__(self):
        if not AnggaranHarian._db_setup_done:
            if database.setup_database_initial():
                AnggaranHarian._db_setup_done = True

    def tambah_transaksi(self, transaksi: Transaksi) -> bool:
        if not isinstance(transaksi, Transaksi) or transaksi.jumlah <= 0: return False
        sql = "INSERT INTO transaksi (deskripsi, jumlah, kategori, tanggal) VALUES (?, ?, ?, ?)"
        params = (transaksi.deskripsi, transaksi.jumlah, transaksi.kategori, transaksi.tanggal.strftime("%Y-%m-%d"))
        last_id = database.execute_query(sql, params)
        if last_id is not None:
            transaksi.id = last_id
            return True
        return False

    def hapus_transaksi(self, id_transaksi: int) -> bool:
        sql = "DELETE FROM transaksi WHERE id = ?"
        hasil = database.execute_query(sql, (id_transaksi,))
        return hasil is not None

    def get_dataframe_transaksi(self, filter_tanggal: datetime.date | None) -> pd.DataFrame:
        query = "SELECT id, tanggal, kategori, deskripsi, jumlah FROM transaksi"
        params = None
        if filter_tanggal:
            query += " WHERE tanggal = ?"
            params = (filter_tanggal.strftime("%Y-%m-%d"),)
        query += " ORDER BY tanggal DESC, id DESC"
        
        df = database.get_dataframe(query, params=params)
        if not df.empty:
            df['Jumlah (Rp)'] = df['jumlah'].map(lambda x: f"Rp {x or 0:,.0f}".replace(",", "."))
            df = df[['id', 'tanggal', 'kategori', 'deskripsi', 'Jumlah (Rp)']]
        return df

    def hitung_total_pengeluaran(self, tanggal: datetime.date | None = None) -> float:
        sql = "SELECT SUM(jumlah) FROM transaksi"
        params = (tanggal.strftime("%Y-%m-%d"),) if tanggal else None
        result = database.fetch_query(sql, params=params, fetch_all=False)
        return float(result[0]) if result and result[0] is not None else 0.0

    def get_pengeluaran_per_kategori(self, tanggal: datetime.date | None = None) -> dict:
        hasil = {}
        sql = "SELECT kategori, SUM(jumlah) FROM transaksi"
        params = [tanggal.strftime("%Y-%m-%d")] if tanggal else []
        sql += " GROUP BY kategori HAVING SUM(jumlah) > 0 ORDER BY SUM(jumlah) DESC"
        rows = database.fetch_query(sql, params=tuple(params) if params else None, fetch_all=True)
        if rows:
            for row in rows:
                hasil[row['kategori'] or "Lainnya"] = float(row[1]) if row[1] is not None else 0.0
        return hasil

Overwriting manajer_anggaran.py


In [12]:
%%writefile main_app.py
import streamlit as st
import datetime
import pandas as pd
from model import Transaksi
from manajer_anggaran import AnggaranHarian
from konfigurasi import KATEGORI_PENGELUARAN

def format_rp(angka):
    return f"Rp {angka or 0:,.0f}".replace(",", ".")

st.set_page_config(page_title="Catatan Pengeluaran", layout="wide")

@st.cache_resource
def get_anggaran_manager(): return AnggaranHarian()
anggaran = get_anggaran_manager()

def halaman_input(anggaran: AnggaranHarian):
    st.header("Tambah Pengeluaran Baru")
    with st.form("form_transaksi_baru", clear_on_submit=True):
        c1, c2 = st.columns([3, 1])
        deskripsi = c1.text_input("Deskripsi*")
        kategori = c2.selectbox("Kategori*:", KATEGORI_PENGELUARAN)
        c3, c4 = st.columns([1, 1])
        jumlah = c3.number_input("Jumlah (Rp)*:", min_value=0.01)
        tanggal = c4.date_input("Tanggal*:", value=datetime.date.today())
        
        if st.form_submit_button("Simpan Transaksi"):
            if deskripsi and jumlah > 0:
                tx = Transaksi(deskripsi, float(jumlah), kategori, tanggal)
                if anggaran.tambah_transaksi(tx):
                    st.success("Tersimpan!")
                    st.cache_data.clear()
                    st.rerun()
            else: st.warning("Data tidak lengkap!")

def halaman_riwayat(anggaran: AnggaranHarian):
    st.subheader("Detail Semua Transaksi")
    df = anggaran.get_dataframe_transaksi(None)
    if not df.empty:
        st.dataframe(df, hide_index=True, use_container_width=True)
        st.divider()
        st.markdown("### 🗑️ Hapus Transaksi")
        with st.form("form_hapus"):
            id_hapus = st.number_input("Masukkan ID Transaksi yang dihapus:", min_value=1, step=1)
            if st.form_submit_button("Hapus Transaksi"):
                if anggaran.hapus_transaksi(id_hapus):
                    st.success("Terhapus!")
                    st.cache_data.clear()
                    st.rerun()

def halaman_ringkasan(anggaran: AnggaranHarian):
    st.subheader("Ringkasan Pengeluaran")
    total = anggaran.hitung_total_pengeluaran()
    st.metric(label="Total Pengeluaran (Semua Waktu)", value=format_rp(total))
    
    st.divider()
    dict_kat = anggaran.get_pengeluaran_per_kategori()
    if dict_kat:
        df_kat = pd.DataFrame([{"Kategori": k, "Total": v} for k, v in dict_kat.items()])
        st.bar_chart(df_kat.set_index('Kategori')['Total'])

menu = st.sidebar.radio("Pilih Menu:", ["Tambah", "Riwayat", "Ringkasan"])
if menu == "Tambah": halaman_input(anggaran)
elif menu == "Riwayat": halaman_riwayat(anggaran)
elif menu == "Ringkasan": halaman_ringkasan(anggaran)

Overwriting main_app.py
